# 原始 FT-900 半导体出口与 Stage1 决策日

原始 PDF 当前/同版上月出口值计算百分比月变动；2012-01 分类过渡隔离。按印出的 08:30 美东发布日期延至下一真实 Qlib 交易日 09:31，映射到冻结 Stage1 训练/留出信号。来源与时序审查，零 SOXX 结果、拟合与账户。

In [1]:
from pathlib import Path
import hashlib,json,re,os
import numpy as np,pandas as pd,fitz,qlib
from qlib.data import D
from qlib.data.dataset import DatasetH
from qlib.data.dataset.handler import DataHandlerLP
from qlib.workflow import R
from src.data.minute_intake import atomic_json
from src.data.nas_market_data import archive_nas_recorder
root=Path.cwd();card=root/'research/census_ft900_stage1_availability_20261023';plan_path=card/'plan.json';plan=json.loads(plan_path.read_text());spec=plan['source_cards']
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
repair=json.loads((card/'header_spacing_diagnostic.json').read_text());prior=json.loads((card/'run_state.json').read_text())
assert plan['frozen'] and plan['execution_authorized'] and prior['status']=='failed_pre_Qlib_header_parser'
assert repair['status']=='all_44_headers_parse_with_optional_comma_space' and repair['old_grammar_matches']==43 and repair['optional_space_matches']==44
for name in ('training_rows','held_rows','lead_in_rows','transition_adjudication'):
    assert sha(root/spec[name])==spec[name+'_sha256'],name
assert sha(root/plan['Qlib_calendar_provider']/'qlib_store_manifest.json')==plan['Qlib_calendar_manifest_sha256']
train_data=json.loads((root/spec['training_rows']).read_text());held_data=json.loads((root/spec['held_rows']).read_text());lead_data=json.loads((root/spec['lead_in_rows']).read_text());transition=json.loads((root/spec['transition_adjudication']).read_text())
assert len(train_data['rows'])==144 and train_data['unique_numeric_Exhibit7_rows']==144
assert len(held_data['rows'])==44 and held_data['unique_numeric_Exhibit7_rows']==44 and held_data['matching_report_months']==44
assert len(lead_data['rows'])==3 and transition['decision'].startswith('Exclude January 2012')
atomic_json({'status':'repair_attempted','plan_sha256':sha(plan_path),'attempt1_held_PDF_opens':20,'diagnostic_held_PDF_opens':44,'completion_held_PDF_open_attempts':44,'D_calendar_attempts':1,'DatasetH_attempts':1,'model_fits':0,'native_accounts':0},card/'run_state.json')
month_pattern=r'\b(?:JANUARY|FEBRUARY|MARCH|APRIL|MAY|JUNE|JULY|AUGUST|SEPTEMBER|OCTOBER|NOVEMBER|DECEMBER)\s+\d{1,2},\s*\d{4}\b'
def parse_release(header):
    assert re.search(r'8:30\s*(?:A\.?M\.?|AM)',header,re.I),header
    date=re.search(month_pattern,header,re.I);assert date,header
    zone=re.search(r'\b(?:EST|EDT)\b',header,re.I);assert zone,header
    return pd.Timestamp(pd.to_datetime(date.group(0))).normalize(),zone.group(0).upper()
source_rows=[]
for x in lead_data['rows']:
    release,zone=parse_release(x['release_header'])
    source_rows.append({'observation_month':x['reference_month'],'release_date':release,'release_zone':zone,'release_header':x['release_header'],'current_million_usd':x['current_month_million_usd'],'prior_same_release_million_usd':x['prior_month_same_release_million_usd'],'pdf_sha256':x['source_sha256'],'source_kind':'2010_lead_in'})
for x in train_data['rows']:
    release,zone=parse_release(x['printed_release_header'])
    source_rows.append({'observation_month':x['reference_month'],'release_date':release,'release_zone':zone,'release_header':x['printed_release_header'],'current_million_usd':x['current_month_million_usd'],'prior_same_release_million_usd':x['prior_month_same_release_million_usd'],'pdf_sha256':x['source_pdf_sha256'],'source_kind':'training'})
for x in held_data['rows']:
    path=root/'data/snapshots/census_ft900_held_20261023'/x['source_url'].rsplit('/',1)[-1]
    payload=path.read_bytes();assert len(payload)==x['source_bytes'] and hashlib.sha256(payload).hexdigest()==x['source_pdf_sha256']
    first=' '.join(fitz.open(stream=payload,filetype='pdf')[0].get_text().split())
    header=re.search(r'FOR\s+(?:IMMEDIATE\s+)?RELEASE.{0,180}?(?:19|20)\d{2}',first,re.I);assert header,x['reference_month']
    release,zone=parse_release(header.group(0))
    source_rows.append({'observation_month':x['reference_month'],'release_date':release,'release_zone':zone,'release_header':header.group(0),'current_million_usd':x['current_month_million_usd'],'prior_same_release_million_usd':x['prior_month_same_release_million_usd'],'pdf_sha256':x['source_pdf_sha256'],'source_kind':'held_reused'})
events=pd.DataFrame(source_rows).sort_values('observation_month').reset_index(drop=True)
assert len(events)==191 and events.observation_month.is_unique and events.release_date.is_unique and events.release_date.is_monotonic_increasing
assert (events.current_million_usd>0).all() and (events.prior_same_release_million_usd>0).all()
events['quarantined_definition_transition']=events.observation_month.eq('2012-01')
assert events.quarantined_definition_transition.sum()==1
events['export_mom_pct_same_vintage']=(events.current_million_usd/events.prior_same_release_million_usd-1)*100
events.loc[events.quarantined_definition_transition,'export_mom_pct_same_vintage']=np.nan
events['definition_regime']=np.where(events.observation_month<'2012-01','pre_2012','post_2012')
usable=events.loc[~events.quarantined_definition_transition].reset_index(drop=True)
assert len(usable)==190 and usable.export_mom_pct_same_vintage.notna().all()
os.environ.setdefault('MLFLOW_ALLOW_FILE_STORE','true')
qlib.init(provider_uri=str(root/plan['Qlib_calendar_provider']),region='us',kernels=1)
cal=pd.DatetimeIndex(D.calendar(start_time='2010-09-01',end_time='2026-10-07',freq='day')).normalize()
assert cal.is_unique and cal.is_monotonic_increasing
bounds=plan['signal_bounds'];train=cal[(cal>=pd.Timestamp(bounds['train'][0]))&(cal<=pd.Timestamp(bounds['train'][1]))];held=cal[(cal>=pd.Timestamp(bounds['held_reused'][0]))&(cal<=pd.Timestamp(bounds['held_reused'][1]))]
assert len(train)==bounds['train_expected']==3020 and len(held)==bounds['held_expected']==944
signals=train.append(held)
release_dates=pd.DatetimeIndex(usable.release_date);positions=cal.searchsorted(release_dates,side='right')
assert (positions<len(cal)).all()
available=cal[positions];assert available.is_monotonic_increasing and not available.has_duplicates
lookup=available.searchsorted(signals,side='right')-1;assert (lookup>=0).all()
chosen=usable.iloc[lookup].reset_index(drop=True);ages=np.asarray((signals-pd.DatetimeIndex(chosen.release_date)).days,dtype=int)
assert (ages>=1).all() and (pd.DatetimeIndex(chosen.release_date)<signals).all()
full=pd.DataFrame({'signal_date':signals,'export_mom_pct_same_vintage':chosen.export_mom_pct_same_vintage.to_numpy(float),'release_age_days':ages,'source_release_date':chosen.release_date.to_numpy(),'source_observation_month':chosen.observation_month.to_numpy(),'source_pdf_sha256':chosen.pdf_sha256.to_numpy(),'definition_regime':chosen.definition_regime.to_numpy(),'historical_first_public_bytes_qualified':False})
assert len(full)==3964 and full.export_mom_pct_same_vintage.notna().all()
feature=full.set_index('signal_date')[['export_mom_pct_same_vintage','release_age_days']].copy();feature['definition_post_2012']=(full.definition_regime.to_numpy()=='post_2012').astype(int);feature['instrument']='SOXX_FT900_SEMI_EXPORT';feature.index.name='datetime';feature=feature.reset_index().set_index(['datetime','instrument'])
panel=pd.concat({'feature':feature},axis=1);dataset=DatasetH(handler=DataHandlerLP.from_df(panel),segments={'all':(None,None)});prepared=dataset.prepare('all',col_set=['feature']);pd.testing.assert_frame_equal(prepared,panel)
annual=full.groupby(full.signal_date.dt.year).agg(signals=('signal_date','size'),mean_age_days=('release_age_days','mean'),maximum_age_days=('release_age_days','max'),distinct_releases=('source_release_date','nunique')).reset_index().rename(columns={'signal_date':'year'});annual.to_csv(card/'yearly_availability.csv',index=False)
audit={'status':'FT900_source_calendar_alignment_complete_information_and_first_public_unqualified','plan_sha256':sha(plan_path),'source_events':len(events),'usable_source_events':len(usable),'quarantined_observation_month':'2012-01','native_Qlib_calendar_rows':len(cal),'train_signal_rows':len(train),'held_reused_signal_rows':len(held),'mapped_source_rows':len(full),'missing_train_rows':0,'missing_held_rows':0,'first_train_source_month':full.iloc[0].source_observation_month,'last_held_source_month':full.iloc[-1].source_observation_month,'minimum_release_age_days':int(ages.min()),'maximum_release_age_days':int(ages.max()),'same_day_release_use':0,'header_spacing_repair_diagnostic_sha256':sha(card/'header_spacing_diagnostic.json'),'historical_first_public_bytes_qualified':False,'classification_continuity_qualified':False,'market_outcome_reads':0,'model_fits':0,'native_accounts':0,'SOXX_goal_achieved':False}
tracking=card/'qlib_recorder'
with R.start(experiment_name=plan['experiment'],uri=str(tracking)):
    recorder=R.get_recorder();recorder.save_objects(artifact_path='inputs',plan=plan)
    recorder.save_objects(artifact_path='sources',vintage_events=events,available_sessions=pd.DataFrame({'observation_month':usable.observation_month,'release_date':usable.release_date,'available_session':available}))
    recorder.save_objects(artifact_path='analysis',mapped_signals=full,prepared_feature=prepared,availability_audit=audit)
    recorder.log_metrics(mapped_signals=len(full),new_fits=0,native_accounts=0)
    rid,eid=recorder.id,str(recorder.experiment_id)
atomic_json({'recorder_id':rid,'entries':[{'path':str(p.relative_to(card)),'sha256':sha(p),'bytes':p.stat().st_size} for p in sorted((tracking/eid/rid/'artifacts').rglob('*')) if p.is_file()]},card/'verification_manifest.json')
audit.update(recorder_id=rid,experiment_id=eid);atomic_json(audit,card/'availability_audit.json')
atomic_json({'source_id':plan['experiment'],'source_plan':'plan.json','as_of':'2026-10-07'},card/'nas_binding.json')
nas=archive_nas_recorder(card/'nas_binding.json',workspace_root=root,card=card,recorder_id=rid,experiment_id=eid)
summary={'experiment':plan['experiment'],'decision':audit['status'],'objective_achieved':False,'result':'availability_audit.json','recorder_id':rid,'experiment_id':eid,'recorder_uri':str(tracking),'recorder_local_dir':str(tracking/eid/rid),'nas_evidence':nas,'new_model_fits':0,'new_native_accounts':0,'artifact_policy':{'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['inputs/*','sources/*','analysis/*']}}
atomic_json(summary,card/'summary.json');atomic_json({'status':'completed_after_mechanical_header_repair','plan_sha256':sha(plan_path),'attempt1_source_JSON_loads':4,'completion_source_JSON_loads':4,'attempt1_held_PDF_opens':20,'diagnostic_held_PDF_opens':44,'completion_held_PDF_opens':44,'total_held_PDF_opens':108,'D_calendar_calls':1,'DatasetH_builds':1,'recorder_id':rid,'new_fits':0,'accounts':0},card/'run_state.json')
print(json.dumps({'status':audit['status'],'train':len(train),'held':len(held),'max_age':int(ages.max()),'first_source':audit['first_train_source_month'],'last_source':audit['last_held_source_month'],'recorder_id':rid}))

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[38724:MainThread](2026-10-23 00:45:53,568) INFO - qlib.Initialization - [config.py:453] - default_conf: client.


[38724:MainThread](2026-10-23 00:45:53,570) INFO - qlib.Initialization - [__init__.py:82] - qlib successfully initialized based on client settings.


[38724:MainThread](2026-10-23 00:45:53,570) INFO - qlib.Initialization - [__init__.py:84] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/minute_information_pit_allocator_20261012/daily_prices')}


[38724:MainThread](2026-10-23 00:45:53,583) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | Loading data Done


...

/Users/rziiiii/Library/Python/3.14/lib/python/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
[38724:MainThread](2026-10-23 00:45:53,596) WARNING - qlib.workflow - [expm.py:230] - No valid experiment found. Create a new experiment with name census_ft900_stage1_availability_20261023.


[38724:MainThread](2026-10-23 00:45:53,620) INFO - qlib.workflow - [exp.py:258] - Experiment 449371276764721344 starts running ...


[38724:MainThread](2026-10-23 00:45:53,687) INFO - qlib.workflow - [recorder.py:345] - Recorder 0b4c648c44d24561a8c21e0182381aa3 starts running under Experiment 449371276764721344 ...


[38724:MainThread](2026-10-23 00:45:54,277) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | waiting `async_log` Done


{"status": "FT900_source_calendar_alignment_complete_information_and_first_public_unqualified", "train": 3020, "held": 944, "max_age": 76, "first_source": "2010-10", "last_source": "2026-08", "recorder_id": "0b4c648c44d24561a8c21e0182381aa3"}


## 结论边界

完整 Qlib 来源映射只是开发期可用性代理；历史首次公开 PDF 字节和跨分类定义仍未完全合格。任何信息改善仍须单独的冻结检验与完整 0bp/2bp/5bp SOXX 账户及独立前向门。

执行结果：191 个原件事件中隔离 2012-01，190 个可用事件在下一真实 Qlib 交易日 09:31 后映射至训练 3020/3020 和复用留出 944/944 信号，最大来源年龄 76 天。Qlib Recorder `0b4c648c44d24561a8c21e0182381aa3` 与 NAS SHA `5287cd0280623e0e4f607d79bfe8cc3161d67d89dc1c68f1882ddbc14296504e` 经限定读回。首轮日期格式失败及唯一机械修复已留证。历史首发字节、分类连续性、Stage1 增量和 SOXX 账户尚未通过。

后续独立来源复核发现此版本在 2012 分类断点后把旧口径值自动延续至 23 个信号日，与冻结来源裁决冲突。本 Recorder 留作原始审计，不可再作因子输入；规范替代卡 `research/stage1_ft900_export_availability_20261023` 将该 23 日标为未知，见 `source_conflict_correction.json`。